In [14]:
"""
New notebook: figureTableMetadata.ipynb
Run order:  uploadandExtract -> figureTableMetadata -> pageMetadata -> treeBuilder -> ...
 
Generates title/caption/description for every Figure and Table row that was just
inserted (metadata columns are currently empty). Uses OCR text for figures (most
LLMs in this pipeline, e.g. Groq's llama-3.3, are text-only -- OCR + surrounding
page text is the reliable substitute for actually "seeing" the image; swap in a
vision-capable model directly if you have one wired into src.config.llm).
 
Requires: pip install pytesseract pillow   (+ the tesseract binary on PATH)
"""

'\nNew notebook: figureTableMetadata.ipynb\nRun order:  uploadandExtract -> figureTableMetadata -> pageMetadata -> treeBuilder -> ...\n\nGenerates title/caption/description for every Figure and Table row that was just\ninserted (metadata columns are currently empty). Uses OCR text for figures (most\nLLMs in this pipeline, e.g. Groq\'s llama-3.3, are text-only -- OCR + surrounding\npage text is the reliable substitute for actually "seeing" the image; swap in a\nvision-capable model directly if you have one wired into src.config.llm).\n\nRequires: pip install pytesseract pillow   (+ the tesseract binary on PATH)\n'

In [15]:
import json
import pytesseract
from PIL import Image
 
from src.config.db import get_connection
from src.config.llm import llm
 
conn = get_connection()
document_id = "DOC000001"  # match whatever you just ran extraction.py against

In [16]:
def strip_json_fences(text: str) -> str:
    text = text.strip()
    if text.startswith("```json"):
        text = text[7:]
    if text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]
    return text.strip()
 
 
def load_page_text(conn, document_id: str) -> dict:
    with conn.cursor() as cur:
        cur.execute(
            'SELECT "pageNumber", content FROM "Page" WHERE "documentId" = %s',
            (document_id,),
        )
        return {pn: content for pn, content in cur.fetchall()}

In [17]:
def load_figures(conn, document_id: str) -> list:
    with conn.cursor() as cur:
        cur.execute(
            '''
            SELECT f.id, f."figureNumber", f."imagePath", p."pageNumber"
            FROM "Figure" f
            LEFT JOIN "Page" p ON p.id = f."pageId"
            WHERE f."documentId" = %s
            ''',
            (document_id,),
        )
        rows = cur.fetchall()
    return [{"id": r[0], "figureNumber": r[1], "imagePath": r[2], "pageNumber": r[3]} for r in rows]
 
 
def load_tables(conn, document_id: str) -> list:
    with conn.cursor() as cur:
        cur.execute(
            '''
            SELECT t.id, t."tableNumber", t.markdown, p."pageNumber"
            FROM "Table" t
            LEFT JOIN "Page" p ON p.id = t."pageId"
            WHERE t."documentId" = %s
            ''',
            (document_id,),
        )
        rows = cur.fetchall()
    return [{"id": r[0], "tableNumber": r[1], "markdown": r[2], "pageNumber": r[3]} for r in rows]

In [18]:
# Figures
# ---------------------------------------------------------------------------
FIGURE_METADATA_PROMPT = """
You are captioning a figure/image extracted from a document page for a Vectorless RAG system.
 
Page context (surrounding text on the same page):
{page_context}
 
OCR text extracted from inside the image (may be noisy or empty):
{ocr_text}
 
Return ONLY valid JSON, no markdown, no explanation.
Schema:
{{"title": "<short title>", "caption": "<one sentence caption>",
  "description": "<2-3 sentence description of what the figure likely shows>",
  "keywords": ["..."]}}
"""
 
 
def ocr_image(image_path: str) -> str:
    try:
        return pytesseract.image_to_string(Image.open(image_path)).strip()
    except Exception:
        return ""
 
 
def generate_figure_metadata(figure: dict, page_text: dict, llm) -> dict:
    ocr_text = ocr_image(figure["imagePath"])
    page_context = (page_text.get(figure["pageNumber"]) or "")[:1500]
 
    prompt = FIGURE_METADATA_PROMPT.format(page_context=page_context, ocr_text=ocr_text[:1000])
    response = llm.invoke(prompt)
    meta = json.loads(strip_json_fences(response.content))
    meta["ocrText"] = ocr_text
    return meta
 
 
def update_figure_metadata(conn, figure_id: str, meta: dict):
    with conn.cursor() as cur:
        cur.execute(
            '''
            UPDATE "Figure"
            SET title=%s, caption=%s, description=%s, "ocrText"=%s, metadata=%s
            WHERE id=%s
            ''',
            (meta.get("title"), meta.get("caption"), meta.get("description"),
             meta.get("ocrText"), json.dumps(meta), figure_id),
        )
    conn.commit()
 

In [19]:
# Tables
# ---------------------------------------------------------------------------
TABLE_METADATA_PROMPT = """
You are captioning a table extracted from a document page for a Vectorless RAG system.
 
Table (markdown):
{markdown}
 
Page context (surrounding text on the same page):
{page_context}
 
Return ONLY valid JSON, no markdown, no explanation.
Schema:
{{"title": "<short title>", "caption": "<one sentence caption>",
  "description": "<2-3 sentence description of what the table shows>",
  "keywords": ["..."]}}
"""
 
 
def generate_table_metadata(table: dict, page_text: dict, llm) -> dict:
    page_context = (page_text.get(table["pageNumber"]) or "")[:1000]
    prompt = TABLE_METADATA_PROMPT.format(markdown=(table["markdown"] or "")[:2000], page_context=page_context)
    response = llm.invoke(prompt)
    return json.loads(strip_json_fences(response.content))
 
 
def update_table_metadata(conn, table_id: str, meta: dict):
    with conn.cursor() as cur:
        cur.execute(
            'UPDATE "Table" SET title=%s, caption=%s, metadata=%s WHERE id=%s',
            (meta.get("title"), meta.get("caption"), json.dumps(meta), table_id),
        )
    conn.commit()
 

In [20]:
# Run
# ---------------------------------------------------------------------------
page_text = load_page_text(conn, document_id)
 
figures = load_figures(conn, document_id)
for fig in figures:
    meta = generate_figure_metadata(fig, page_text, llm)
    update_figure_metadata(conn, fig["id"], meta)
print(f"Captioned {len(figures)} figures")
 
tables = load_tables(conn, document_id)
for tbl in tables:
    meta = generate_table_metadata(tbl, page_text, llm)
    update_table_metadata(conn, tbl["id"], meta)
print(f"Captioned {len(tables)} tables")

Captioned 16 figures
Captioned 51 tables
